<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/piolotnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# runpod connect
# 2. 환경 확인 및 설정# GPU 확인
!nvidia-smi

# Python 환경 확인
!python --version

# 필요한 라이브러리 설치
!pip install opencv-python matplotlib pandas scikit-learn

In [ ]:
# 3. 작업 디렉토리 생성
!cd /workspace
!mkdir pilotnet_project
!cd pilotnet_project

In [ ]:
import cv2
import pandas as pd
import numpy as np
import os

# 영상 파일 경로
video_file = "/workspace/pilotnet_video.mp4"

# 영상 정보 확인
cap = cv2.VideoCapture(video_file)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = frame_count / fps

print(f"영상 정보:")
print(f"- FPS: {fps}")
print(f"- 총 프레임: {frame_count}")
print(f"- 길이: {duration:.1f}초")

cap.release()

# 폴더 생성
os.makedirs('data', exist_ok=True)
os.makedirs('data/IMG', exist_ok=True)

print("폴더 생성 완료!")

In [ ]:
# 영상에서 프레임 추출
cap = cv2.VideoCapture(video_file)
frame_idx = 0
extracted_frames = 0
data = []

print("프레임 추출 중...")

# 매 15프레임마다 추출 (30fps라면 0.5초마다)
frame_interval = 15

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 지정된 간격마다 프레임 저장
    if frame_idx % frame_interval == 0:
        img_name = f"frame_{extracted_frames:06d}.jpg"
        img_path = f"data/IMG/{img_name}"

        # 이미지 크기 조정 (PilotNet 입력 크기에 맞게)
        resized_frame = cv2.resize(frame, (320, 160))
        cv2.imwrite(img_path, resized_frame)

        # 데이터 기록 (조향각은 일단 0으로 설정)
        data.append({
            'center': f"IMG/{img_name}",
            'steering': 0.0,
            'throttle': 0.5,
            'brake': 0.0,
            'speed': 30.0
        })

        extracted_frames += 1

        if extracted_frames % 50 == 0:
            print(f"{extracted_frames}개 프레임 추출됨...")

    frame_idx += 1

cap.release()

print(f"\n완료! 총 {extracted_frames}개 프레임 추출")

# CSV 파일 생성
df = pd.DataFrame(data)
df.to_csv('data/driving_log.csv', index=False)
print("driving_log.csv 생성 완료!")

In [ ]:
# 추출된 이미지 확인
import matplotlib.pyplot as plt

print(f"총 {len(data)}개의 이미지가 추출되었습니다.")

# 처음 6장 확인
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i in range(6):
    if i < len(data):
        img_path = f"data/{data[i]['center']}"
        if os.path.exists(img_path):
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[i//3, i%3].imshow(img)
            axes[i//3, i%3].set_title(f"Frame {i}")
            axes[i//3, i%3].axis('off')

plt.tight_layout()
plt.show()

print("이제 조향각 라벨링이 필요합니다!")

In [ ]:
# 곡선이 있는 주행을 위한 조향각 설정
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('data/driving_log.csv')
total_frames = len(df)

print(f"총 {total_frames}개 프레임")
print("구간별 조향각을 설정합니다:")

# 구간 설정 (프레임 번호 기준)
# 예시: 0-30% 직선, 30-50% 좌곡선, 50-80% 직선, 80-100% 우곡선
def set_steering_by_sections():
    steering_values = np.zeros(total_frames)

    print("구간을 설정하세요 (프레임 번호 또는 비율):")
    print("1. 직선 구간")
    print("2. 좌곡선 구간")
    print("3. 우곡선 구간")
    print("4. 완료")

    while True:
        choice = input("\n선택 (1-4): ")

        if choice == '1':  # 직선
            start = int(input("시작 프레임 (0부터): "))
            end = int(input("끝 프레임: "))
            # 직선: 평균 0, 약간의 노이즈
            steering_values[start:end] = np.random.normal(0.0, 0.03, end-start)
            steering_values[start:end] = np.clip(steering_values[start:end], -0.1, 0.1)
            print(f"프레임 {start}-{end}: 직선 구간 설정")

        elif choice == '2':  # 좌곡선
            start = int(input("시작 프레임: "))
            end = int(input("끝 프레임: "))
            intensity = float(input("곡선 강도 (-0.2~-0.8, 강할수록 큰 값): ") or "-0.4")
            # 좌곡선: 음수 값
            steering_values[start:end] = np.random.normal(intensity, 0.1, end-start)
            steering_values[start:end] = np.clip(steering_values[start:end], -1.0, 0.0)
            print(f"프레임 {start}-{end}: 좌곡선 구간 설정 (강도: {intensity})")

        elif choice == '3':  # 우곡선
            start = int(input("시작 프레임: "))
            end = int(input("끝 프레임: "))
            intensity = float(input("곡선 강도 (0.2~0.8, 강할수록 큰 값): ") or "0.4")
            # 우곡선: 양수 값
            steering_values[start:end] = np.random.normal(intensity, 0.1, end-start)
            steering_values[start:end] = np.clip(steering_values[start:end], 0.0, 1.0)
            print(f"프레임 {start}-{end}: 우곡선 구간 설정 (강도: {intensity})")

        elif choice == '4':
            break
        else:
            print("잘못된 선택입니다.")

    return steering_values

# 조향각 설정 실행
steering_values = set_steering_by_sections()

# 데이터프레임에 적용
df['steering'] = steering_values

# 결과 확인
print(f"\n조향각 통계:")
print(f"평균: {df['steering'].mean():.3f}")
print(f"최소: {df['steering'].min():.3f}")
print(f"최대: {df['steering'].max():.3f}")

# 저장
df.to_csv('data/driving_log.csv', index=False)
print("조향각 설정 완료!")

# 시각화
plt.figure(figsize=(12, 4))
plt.plot(df['steering'], linewidth=2)
plt.title('설정된 조향각 프로필')
plt.xlabel('프레임')
plt.ylabel('조향각')
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# 영상 패턴에 맞는 조향각 자동 설정
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('data/driving_log.csv')
total_frames = len(df)

print(f"총 {total_frames}개 프레임")
print("영상 패턴: 1분 14초부터 10초간 곡선")

# 프레임 계산 (대략 15fps로 추출했다고 가정)
fps_extracted = total_frames / (1*60 + 24)  # 1분 24초 영상에서 239프레임
curve_start_frame = int(74 * fps_extracted)  # 74초 지점
curve_end_frame = int(84 * fps_extracted)    # 84초 지점

print(f"곡선 구간: 프레임 {curve_start_frame} ~ {curve_end_frame}")

# 조향각 설정
steering_values = np.zeros(total_frames)

# 1. 직선 구간 (처음~곡선 전)
steering_values[0:curve_start_frame] = np.random.normal(0.0, 0.03, curve_start_frame)
steering_values[0:curve_start_frame] = np.clip(steering_values[0:curve_start_frame], -0.1, 0.1)

# 2. 곡선 구간 (우곡선으로 가정, 좌곡선이면 -0.4로 변경)
curve_intensity = 0.4  # 우곡선, 좌곡선이면 -0.4
curve_length = curve_end_frame - curve_start_frame
steering_values[curve_start_frame:curve_end_frame] = np.random.normal(curve_intensity, 0.1, curve_length)
steering_values[curve_start_frame:curve_end_frame] = np.clip(steering_values[curve_start_frame:curve_end_frame], 0.0, 0.8)

# 3. 직선 구간 (곡선 후~끝)
remaining_frames = total_frames - curve_end_frame
steering_values[curve_end_frame:] = np.random.normal(0.0, 0.03, remaining_frames)
steering_values[curve_end_frame:] = np.clip(steering_values[curve_end_frame:], -0.1, 0.1)

# 데이터프레임에 적용
df['steering'] = steering_values

print(f"\n조향각 통계:")
print(f"평균: {df['steering'].mean():.3f}")
print(f"최소: {df['steering'].min():.3f}")
print(f"최대: {df['steering'].max():.3f}")

# 저장
df.to_csv('data/driving_log.csv', index=False)
print("조향각 설정 완료!")

# 시각화
plt.figure(figsize=(12, 4))
plt.plot(steering_values, linewidth=2)
plt.axvline(x=curve_start_frame, color='r', linestyle='--', alpha=0.7, label='곡선 시작')
plt.axvline(x=curve_end_frame, color='r', linestyle='--', alpha=0.7, label='곡선 끝')
plt.title('설정된 조향각 프로필')
plt.xlabel('프레임')
plt.ylabel('조향각')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
plt.show()

In [ ]:
# PilotNet 모델 정의
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

class PilotNet(nn.Module):
    def __init__(self):
        super(PilotNet, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 24, kernel_size=5, stride=2)
        self.conv2 = nn.Conv2d(24, 36, kernel_size=5, stride=2)
        self.conv3 = nn.Conv2d(36, 48, kernel_size=5, stride=2)
        self.conv4 = nn.Conv2d(48, 64, kernel_size=3)
        self.conv5 = nn.Conv2d(64, 64, kernel_size=3)

        # Fully connected layers (크기는 실제 입력에 맞게 자동 계산)
        self.fc1 = nn.Linear(64 * 2 * 17, 100)  # 320x160 입력 기준
        self.fc2 = nn.Linear(100, 50)
        self.fc3 = nn.Linear(50, 10)
        self.fc4 = nn.Linear(10, 1)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = torch.relu(self.conv4(x))
        x = torch.relu(self.conv5(x))

        x = x.view(x.size(0), -1)  # Flatten

        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)

        return x

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 데이터셋 클래스
class DrivingDataset(Dataset):
    def __init__(self, csv_file, img_dir):
        self.driving_log = pd.read_csv(csv_file)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.driving_log)

    def __getitem__(self, idx):
        # 이미지 경로
        img_path = self.driving_log.iloc[idx]['center'].strip()
        img_path = img_path.replace('IMG/', self.img_dir + '/')

        # 이미지 로드 및 전처리
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (200, 66))  # PilotNet 표준 크기

        # 정규화
        image = image / 255.0
        image = np.transpose(image, (2, 0, 1))  # HWC to CHW

        # 조향각
        steering = float(self.driving_log.iloc[idx]['steering'])

        return torch.FloatTensor(image), torch.FloatTensor([steering])

# 데이터셋 생성
dataset = DrivingDataset('data/driving_log.csv', 'data/IMG')
print(f"총 데이터 개수: {len(dataset)}")

# train/validation 분할
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# 데이터 로더
batch_size = 16  # 작은 데이터셋이므로 작은 배치 사이즈
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train 데이터: {len(train_dataset)}")
print(f"Validation 데이터: {len(val_dataset)}")

In [ ]:
# 모델, 손실함수, 옵티마이저 초기화
model = PilotNet().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("모델 파라미터 수:", sum(p.numel() for p in model.parameters()))

# 학습
num_epochs = 20  # 작은 데이터셋이므로 더 많은 에포크
train_losses = []
val_losses = []

print("학습 시작!")
print("-" * 50)

for epoch in range(num_epochs):
    # 학습 모드
    model.train()
    train_loss = 0.0

    for batch_idx, (images, steering) in enumerate(train_loader):
        images, steering = images.to(device), steering.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, steering)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # 검증 모드
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, steering in val_loader:
            images, steering = images.to(device), steering.to(device)
            outputs = model(images)
            loss = criterion(outputs, steering)
            val_loss += loss.item()

    # 평균 손실 계산
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # 진행 상황 출력

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os

print("라이브러리 재로딩 완료!")
print(f"NumPy 버전: {np.__version__}")
print(f"PyTorch 버전: {torch.__version__}")

In [ ]:
# 데이터셋 클래스 다시 정의
class DrivingDataset(Dataset):
    def __init__(self, csv_file, img_dir):
        self.driving_log = pd.read_csv(csv_file)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.driving_log)

    def __getitem__(self, idx):
        # 이미지 경로
        img_path = self.driving_log.iloc[idx]['center'].strip()
        img_path = img_path.replace('IMG/', self.img_dir + '/')

        # 이미지 로드 및 전처리
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (200, 66))  # PilotNet 표준 크기

        # 정규화
        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))  # HWC to CHW

        # 조향각
        steering = float(self.driving_log.iloc[idx]['steering'])

        return torch.FloatTensor(image), torch.FloatTensor([steering])

# 데이터셋 다시 생성
dataset = DrivingDataset('data/driving_log.csv', 'data/IMG')
print(f"총 데이터 개수: {len(dataset)}")

# train/validation 분할
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # 재현 가능한 결과
)

# 데이터 로더 (worker 수를 0으로 설정)
batch_size = 8  # 더 작은 배치 사이즈로 시도
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Train 데이터: {len(train_dataset)}")
print(f"Validation 데이터: {len(val_dataset)}")
print("데이터 로더 생성 완료!")

In [ ]:
# 모델, 손실함수, 옵티마이저 초기화
model = PilotNet().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("모델 파라미터 수:", sum(p.numel() for p in model.parameters()))

# 학습
num_epochs = 20  # 작은 데이터셋이므로 더 많은 에포크
train_losses = []
val_losses = []

print("학습 시작!")
print("-" * 50)

for epoch in range(num_epochs):
    # 학습 모드
    model.train()
    train_loss = 0.0

    for batch_idx, (images, steering) in enumerate(train_loader):
        images, steering = images.to(device), steering.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, steering)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # 검증 모드
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, steering in val_loader:
            images, steering = images.to(device), steering.to(device)
            outputs = model(images)
            loss = criterion(outputs, steering)
            val_loss += loss.item()

    # 평균 손실 계산
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # 진행 상황 출력 (이 부분을 추가하세요!)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1:2d}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

print("-" * 50)
print("학습 완료!")

# 모델 저장
torch.save(model.state_dict(), 'pilotnet_model.pth')
print("모델 저장 완료: pilotnet_model.pth")

In [ ]:
# 위에서  에러나서 다시 함
# 1. 모든 라이브러리 다시 설치 및 import
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"NumPy 버전: {np.__version__}")
print(f"PyTorch 버전: {torch.__version__}")

In [ ]:
# 데이터 직접 로딩 (DataLoader 문제 우회)
df = pd.read_csv('data/driving_log.csv')
print(f"데이터 개수: {len(df)}")

# 수동으로 train/val 분할
train_df = df[:int(0.8 * len(df))]
val_df = df[int(0.8 * len(df)):]

print(f"Train: {len(train_df)}, Val: {len(val_df)}")

# 데이터 로딩 함수
def load_data_batch(dataframe, batch_size, start_idx):
    batch_images = []
    batch_steering = []

    for i in range(start_idx, min(start_idx + batch_size, len(dataframe))):
        # 이미지 로드
        img_path = f"data/{dataframe.iloc[i]['center']}"
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (200, 66))
        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))

        # 조향각
        steering = float(dataframe.iloc[i]['steering'])

        batch_images.append(image)
        batch_steering.append([steering])

    return torch.FloatTensor(batch_images), torch.FloatTensor(batch_steering)

# 테스트
test_images, test_steering = load_data_batch(train_df, 4, 0)
print(f"테스트 배치 형태: {test_images.shape}, {test_steering.shape}")

In [ ]:
class PilotNet(nn.Module):
    def __init__(self):
        super(PilotNet, self).__init__()

        self.conv1 = nn.Conv2d(3, 24, kernel_size=5, stride=2)
        self.conv2 = nn.Conv2d(24, 36, kernel_size=5, stride=2)
        self.conv3 = nn.Conv2d(36, 48, kernel_size=5, stride=2)
        self.conv4 = nn.Conv2d(48, 64, kernel_size=3)
        self.conv5 = nn.Conv2d(64, 64, kernel_size=3)

        # 200x66 입력에 맞게 조정된 크기
        self.fc1 = nn.Linear(64 * 1 * 18, 100)
        self.fc2 = nn.Linear(100, 50)
        self.fc3 = nn.Linear(50, 10)
        self.fc4 = nn.Linear(10, 1)

        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = torch.relu(self.conv4(x))
        x = torch.relu(self.conv5(x))

        x = x.view(x.size(0), -1)

        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)

        return x

model = PilotNet().to(device)
print("모델 생성 완료!")

In [ ]:
# 모델, 손실함수, 옵티마이저 초기화
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("모델 파라미터 수:", sum(p.numel() for p in model.parameters()))

# 학습 설정
num_epochs = 15
batch_size = 8
train_losses = []
val_losses = []

print("학습 시작!")
print("-" * 50)

for epoch in range(num_epochs):
    # 학습 모드
    model.train()
    train_loss = 0.0
    train_batches = 0

    # 학습 데이터를 배치별로 처리
    for start_idx in range(0, len(train_df), batch_size):
        # 배치 로드
        images, steering = load_data_batch(train_df, batch_size, start_idx)
        images, steering = images.to(device), steering.to(device)

        # 학습
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, steering)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_batches += 1

    # 검증 모드
    model.eval()
    val_loss = 0.0
    val_batches = 0

    with torch.no_grad():
        for start_idx in range(0, len(val_df), batch_size):
            images, steering = load_data_batch(val_df, batch_size, start_idx)
            images, steering = images.to(device), steering.to(device)

            outputs = model(images)
            loss = criterion(outputs, steering)
            val_loss += loss.item()
            val_batches += 1

    # 평균 손실 계산
    train_loss /= train_batches
    val_loss /= val_batches

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # 진행 상황 출력
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1:2d}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

print("-" * 50)
print("학습 완료!")

# 모델 저장
torch.save(model.state_dict(), 'pilotnet_model.pth')
print("모델 저장 완료: pilotnet_model.pth")

In [ ]:
# 학습 결과 시각화
plt.figure(figsize=(15, 5))

# 1. 손실 그래프
plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Train Loss', linewidth=2, marker='o')
plt.plot(val_losses, label='Validation Loss', linewidth=2, marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)

# 2. 예측 vs 실제 비교
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for start_idx in range(0, len(val_df), 8):
        images, steering = load_data_batch(val_df, 8, start_idx)
        images = images.to(device)
        pred = model(images)
        predictions.extend(pred.cpu().numpy().flatten())
        actuals.extend(steering.numpy().flatten())

plt.subplot(1, 3, 2)
plt.scatter(actuals, predictions, alpha=0.7, s=50)
plt.plot([-1, 1], [-1, 1], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('실제 조향각')
plt.ylabel('예측 조향각')
plt.title('예측 vs 실제')
plt.legend()
plt.grid(True, alpha=0.3)

# 3. 예측 오차 분포
errors = np.array(predictions) - np.array(actuals)
plt.subplot(1, 3, 3)
plt.hist(errors, bins=15, alpha=0.7, edgecolor='black', color='skyblue')
plt.xlabel('예측 오차')
plt.ylabel('빈도')
plt.title('예측 오차 분포')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 성능 지표
mse = np.mean(errors**2)
mae = np.mean(np.abs(errors))
rmse = np.sqrt(mse)

print(f"\n🎯 모델 성능 지표:")
print(f"MSE (평균 제곱 오차): {mse:.6f}")
print(f"MAE (평균 절대 오차): {mae:.6f}")
print(f"RMSE (제곱근 평균 제곱 오차): {rmse:.6f}")
print(f"\n최종 Train Loss: {train_losses[-1]:.6f}")
print(f"최종 Validation Loss: {val_losses[-1]:.6f}")

In [ ]:
#  실시간 PilotNet 시뮬레이터 만들기:
#  코드를 실행하면 실시간으로 PilotNet이 예측하는 모습을 볼 수 있습니다!

# 왼쪽 위: 카메라 뷰
# 오른쪽 위: 조향각 게이지 (빨간색=예측, 파란색=실제)
# 왼쪽 아래: 조향각 히스토리 그래프
# 오른쪽 아래: 성능 지표 및 상태
# 실시간 PilotNet 시뮬레이터
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import time

class PilotNetSimulator:
    def __init__(self, model_path='pilotnet_model.pth'):
        # 모델 로드
        self.model = PilotNet().to(device)
        self.model.load_state_dict(torch.load(model_path))
        self.model.eval()

        # 시뮬레이션 데이터
        self.current_frame = 0
        self.steering_history = []
        self.speed = 30  # km/h

        print("PilotNet 시뮬레이터 준비 완료!")

    def preprocess_image(self, image):
        """이미지 전처리"""
        processed = cv2.resize(image, (200, 66))
        processed = processed.astype(np.float32) / 255.0
        processed = np.transpose(processed, (2, 0, 1))
        return torch.FloatTensor(processed).unsqueeze(0).to(device)

    def predict_steering(self, image):
        """조향각 예측"""
        with torch.no_grad():
            processed = self.preprocess_image(image)
            steering = self.model(processed).cpu().item()
            return steering

    def run_simulation(self):
        """시뮬레이션 실행"""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

        # 데이터 로드
        df = pd.read_csv('data/driving_log.csv')

        for frame_idx in range(0, len(df), 3):  # 3프레임마다
            # 현재 이미지 로드
            img_path = f"data/{df.iloc[frame_idx]['center']}"
            image = cv2.imread(img_path)
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # 조향각 예측
            predicted_steering = self.predict_steering(image_rgb)
            actual_steering = float(df.iloc[frame_idx]['steering'])

            # 조향각 히스토리 업데이트
            self.steering_history.append({
                'frame': frame_idx,
                'predicted': predicted_steering,
                'actual': actual_steering
            })

            # 최근 50프레임만 유지
            if len(self.steering_history) > 50:
                self.steering_history.pop(0)

            # 시각화 업데이트
            self.update_display(ax1, ax2, ax3, ax4, image_rgb, predicted_steering, actual_steering, frame_idx, len(df))

            plt.pause(0.1)  # 0.1초 간격

        plt.show()

    def update_display(self, ax1, ax2, ax3, ax4, image, pred_steering, actual_steering, frame, total_frames):
        """디스플레이 업데이트"""
        # 1. 현재 카메라 뷰
        ax1.clear()
        ax1.imshow(image)
        ax1.set_title(f'PilotNet Camera View\nFrame: {frame}/{total_frames}', fontsize=12)
        ax1.axis('off')

        # 2. 조향각 게이지 (원형)
        ax2.clear()
        angles = np.linspace(0, 2*np.pi, 100)
        ax2.plot(np.cos(angles), np.sin(angles), 'k-', linewidth=3)

        # 예측 조향각 화살표 (빨간색)
        pred_angle = pred_steering * np.pi / 2  # -90도 ~ 90도
        ax2.arrow(0, 0, 0.8*np.sin(pred_angle), 0.8*np.cos(pred_angle),
                 head_width=0.1, head_length=0.1, fc='red', ec='red', linewidth=3)

        # 실제 조향각 화살표 (파란색, 얇게)
        actual_angle = actual_steering * np.pi / 2
        ax2.arrow(0, 0, 0.6*np.sin(actual_angle), 0.6*np.cos(actual_angle),
                 head_width=0.05, head_length=0.05, fc='blue', ec='blue', linewidth=2)

        ax2.set_xlim(-1.2, 1.2)
        ax2.set_ylim(-1.2, 1.2)
        ax2.set_aspect('equal')
        ax2.set_title(f'Steering Wheel\nPred: {pred_steering:.3f} (Red)\nActual: {actual_steering:.3f} (Blue)', fontsize=10)
        ax2.axis('off')

        # 3. 조향각 히스토리
        ax3.clear()
        if len(self.steering_history) > 1:
            frames = [h['frame'] for h in self.steering_history]
            predicted = [h['predicted'] for h in self.steering_history]
            actual = [h['actual'] for h in self.steering_history]

            ax3.plot(frames, predicted, 'r-', label='Predicted', linewidth=2)
            ax3.plot(frames, actual, 'b-', label='Actual', linewidth=2)
            ax3.set_ylabel('Steering Angle')
            ax3.set_xlabel('Frame')
            ax3.set_title('Steering History')
            ax3.legend()
            ax3.grid(True, alpha=0.3)

        # 4. 성능 지표
        ax4.clear()
        if len(self.steering_history) > 0:
            recent_pred = [h['predicted'] for h in self.steering_history]
            recent_actual = [h['actual'] for h in self.steering_history]

            errors = [abs(p - a) for p, a in zip(recent_pred, recent_actual)]
            avg_error = np.mean(errors) if errors else 0
            max_error = max(errors) if errors else 0

            # 성능 지표 텍스트
            performance_text = f"""
PilotNet Performance

Current Frame: {frame}
Progress: {frame/total_frames*100:.1f}%

Steering Prediction:
  Current: {pred_steering:.4f}
  Actual: {actual_steering:.4f}
  Error: {abs(pred_steering - actual_steering):.4f}

Recent Performance:
  Avg Error: {avg_error:.4f}
  Max Error: {max_error:.4f}

Vehicle Status:
  Speed: {self.speed} km/h
  Status: {"🟢 AUTONOMOUS" if avg_error < 0.1 else "🟡 LEARNING"}
            """

            ax4.text(0.05, 0.95, performance_text, transform=ax4.transAxes,
                    fontsize=10, verticalalignment='top', fontfamily='monospace')
            ax4.set_xlim(0, 1)
            ax4.set_ylim(0, 1)
            ax4.axis('off')

        plt.tight_layout()

# 시뮬레이터 실행
simulator = PilotNetSimulator()
print("시뮬레이터를 시작합니다...")
simulator.run_simulation()

In [ ]:
# RunPod에 최적화된 실시간 PilotNet 시뮬레이터
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import time

class PilotNetSimulator:
    def __init__(self, model_path='pilotnet_model.pth'):
        # 모델 로드
        self.model = PilotNet().to(device)
        self.model.load_state_dict(torch.load(model_path))
        self.model.eval()

        # 시뮬레이션 데이터
        self.current_frame = 0
        self.steering_history = []
        self.speed = 30  # km/h

        print("🚗 PilotNet 시뮬레이터 준비 완료!")

    def preprocess_image(self, image):
        """이미지 전처리"""
        processed = cv2.resize(image, (200, 66))
        processed = processed.astype(np.float32) / 255.0
        processed = np.transpose(processed, (2, 0, 1))
        return torch.FloatTensor(processed).unsqueeze(0).to(device)

    def predict_steering(self, image):
        """조향각 예측"""
        with torch.no_grad():
            processed = self.preprocess_image(image)
            steering = self.model(processed).cpu().item()
            return steering

    def run_simulation(self):
        """시뮬레이션 실행"""
        # 데이터 로드
        df = pd.read_csv('data/driving_log.csv')

        print("🎮 시뮬레이션 시작!")
        print("=" * 60)

        # matplotlib 백엔드 설정
        plt.ion()  # 인터랙티브 모드 켜기
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('PilotNet Real-time Simulator', fontsize=16)

        try:
            for i, frame_idx in enumerate(range(0, len(df), 5)):  # 5프레임마다 (더 빠르게)
                # 현재 이미지 로드
                img_path = f"data/{df.iloc[frame_idx]['center']}"
                image = cv2.imread(img_path)

                if image is None:
                    continue

                image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                # 조향각 예측
                predicted_steering = self.predict_steering(image_rgb)
                actual_steering = float(df.iloc[frame_idx]['steering'])

                # 조향각 히스토리 업데이트
                self.steering_history.append({
                    'frame': frame_idx,
                    'predicted': predicted_steering,
                    'actual': actual_steering
                })

                # 최근 30프레임만 유지 (메모리 절약)
                if len(self.steering_history) > 30:
                    self.steering_history.pop(0)

                # 시각화 업데이트
                self.update_display(ax1, ax2, ax3, ax4, image_rgb,
                                  predicted_steering, actual_steering, frame_idx, len(df))

                # 진행 상황 출력 (콘솔)
                if i % 10 == 0:
                    progress = frame_idx / len(df) * 100
                    error = abs(predicted_steering - actual_steering)
                    print(f"Frame {frame_idx:3d} | Progress: {progress:5.1f}% | "
                          f"Pred: {predicted_steering:6.3f} | "
                          f"Actual: {actual_steering:6.3f} | "
                          f"Error: {error:.3f}")

                plt.pause(0.2)  # 0.2초 간격 (더 천천히)

        except KeyboardInterrupt:
            print("\n⏹️  시뮬레이션이 사용자에 의해 중단되었습니다.")
        except Exception as e:
            print(f"\n❌ 오류 발생: {e}")
        finally:
            plt.ioff()  # 인터랙티브 모드 끄기
            print("🏁 시뮬레이션 완료!")

    def update_display(self, ax1, ax2, ax3, ax4, image, pred_steering, actual_steering, frame, total_frames):
        """디스플레이 업데이트"""
        # 1. 현재 카메라 뷰
        ax1.clear()
        ax1.imshow(image)
        ax1.set_title(f'🎥 Camera View\nFrame: {frame}/{total_frames}', fontsize=12)
        ax1.axis('off')

        # 2. 조향각 게이지 (원형)
        ax2.clear()
        angles = np.linspace(0, 2*np.pi, 100)
        ax2.plot(np.cos(angles), np.sin(angles), 'k-', linewidth=3)

        # 예측 조향각 화살표 (빨간색)
        pred_angle = pred_steering * np.pi / 2  # -90도 ~ 90도
        ax2.arrow(0, 0, 0.8*np.sin(pred_angle), 0.8*np.cos(pred_angle),
                 head_width=0.1, head_length=0.1, fc='red', ec='red', linewidth=3)

        # 실제 조향각 화살표 (파란색)
        actual_angle = actual_steering * np.pi / 2
        ax2.arrow(0, 0, 0.6*np.sin(actual_angle), 0.6*np.cos(actual_angle),
                 head_width=0.05, head_length=0.05, fc='blue', ec='blue', linewidth=2)

        ax2.set_xlim(-1.2, 1.2)
        ax2.set_ylim(-1.2, 1.2)
        ax2.set_aspect('equal')
        ax2.set_title(f'🎯 Steering\nPred: {pred_steering:.3f} 🔴\nActual: {actual_steering:.3f} 🔵', fontsize=10)
        ax2.axis('off')

        # 3. 조향각 히스토리
        ax3.clear()
        if len(self.steering_history) > 1:
            frames = [h['frame'] for h in self.steering_history]
            predicted = [h['predicted'] for h in self.steering_history]
            actual = [h['actual'] for h in self.steering_history]

            ax3.plot(frames, predicted, 'r-', label='Predicted', linewidth=2, marker='o', markersize=3)
            ax3.plot(frames, actual, 'b-', label='Actual', linewidth=2, marker='s', markersize=3)
            ax3.set_ylabel('Steering Angle')
            ax3.set_xlabel('Frame')
            ax3.set_title('📈 Steering History')
            ax3.legend()
            ax3.grid(True, alpha=0.3)

        # 4. 성능 지표
        ax4.clear()
        if len(self.steering_history) > 0:
            recent_pred = [h['predicted'] for h in self.steering_history]
            recent_actual = [h['actual'] for h in self.steering_history]

            errors = [abs(p - a) for p, a in zip(recent_pred, recent_actual)]
            avg_error = np.mean(errors) if errors else 0
            max_error = max(errors) if errors else 0

            # 성능 지표 텍스트
            status_emoji = "🟢 EXCELLENT" if avg_error < 0.05 else "🟡 GOOD" if avg_error < 0.1 else "🔴 LEARNING"

            performance_text = f"""🤖 PilotNet Performance

📊 Current Status:
   Frame: {frame}
   Progress: {frame/total_frames*100:.1f}%

🎯 Steering Prediction:
   Current: {pred_steering:.4f}
   Actual: {actual_steering:.4f}
   Error: {abs(pred_steering - actual_steering):.4f}

📈 Recent Performance:
   Avg Error: {avg_error:.4f}
   Max Error: {max_error:.4f}

🚗 Vehicle Status:
   Speed: {self.speed} km/h
   Status: {status_emoji}
            """

            ax4.text(0.05, 0.95, performance_text, transform=ax4.transAxes,
                    fontsize=9, verticalalignment='top', fontfamily='monospace')
            ax4.set_xlim(0, 1)
            ax4.set_ylim(0, 1)
            ax4.axis('off')

        plt.tight_layout()

# 시뮬레이터 실행
print("🚀 PilotNet 시뮬레이터를 시작합니다...")
simulator = PilotNetSimulator()
simulator.run_simulation()

In [ ]:
# 위에서 matplotlib에러로 수정한 코드

# 텍스트 기반 PilotNet 실시간 시뮬레이터 (matplotlib 없이)
import cv2
import numpy as np
import torch
import time
import os

class TextPilotNetSimulator:
    def __init__(self, model_path='pilotnet_model.pth'):
        # 모델 로드
        self.model = PilotNet().to(device)
        self.model.load_state_dict(torch.load(model_path))
        self.model.eval()

        self.steering_history = []
        print("🚗 PilotNet 텍스트 시뮬레이터 준비 완료!")

    def preprocess_image(self, image):
        """이미지 전처리"""
        processed = cv2.resize(image, (200, 66))
        processed = processed.astype(np.float32) / 255.0
        processed = np.transpose(processed, (2, 0, 1))
        return torch.FloatTensor(processed).unsqueeze(0).to(device)

    def predict_steering(self, image):
        """조향각 예측"""
        with torch.no_grad():
            processed = self.preprocess_image(image)
            steering = self.model(processed).cpu().item()
            return steering

    def draw_steering_wheel(self, predicted, actual):
        """ASCII로 조향각 표시"""
        # -1~1 범위를 -10~10으로 스케일
        pred_pos = int(predicted * 10)
        actual_pos = int(actual * 10)

        # 조향 휠 시각화 (21자리)
        wheel = [' '] * 21
        center = 10

        # 실제값 (파란색 대신 A)
        if -10 <= actual_pos <= 10:
            wheel[center + actual_pos] = 'A'

        # 예측값 (빨간색 대신 P)
        if -10 <= pred_pos <= 10:
            if wheel[center + pred_pos] == 'A':
                wheel[center + pred_pos] = '*'  # 겹치는 경우
            else:
                wheel[center + pred_pos] = 'P'

        # 중앙점
        if wheel[center] == ' ':
            wheel[center] = '|'

        return ''.join(wheel)

    def run_simulation(self):
        """시뮬레이션 실행"""
        df = pd.read_csv('data/driving_log.csv')

        print("\n🎮 PilotNet 실시간 시뮬레이션 시작!")
        print("=" * 80)
        print("범례: P=예측값, A=실제값, *=일치, |=중앙")
        print("좌←                    |중앙|                    →우")
        print("=" * 80)

        total_error = 0
        frame_count = 0

        try:
            for frame_idx in range(0, len(df), 3):  # 3프레임마다
                # 현재 이미지 로드
                img_path = f"data/{df.iloc[frame_idx]['center']}"
                image = cv2.imread(img_path)

                if image is None:
                    continue

                image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                # 조향각 예측
                predicted_steering = self.predict_steering(image_rgb)
                actual_steering = float(df.iloc[frame_idx]['steering'])
                error = abs(predicted_steering - actual_steering)

                # 통계 업데이트
                total_error += error
                frame_count += 1
                avg_error = total_error / frame_count

                # 조향각 히스토리 업데이트
                self.steering_history.append({
                    'predicted': predicted_steering,
                    'actual': actual_steering,
                    'error': error
                })

                # 최근 10프레임만 유지
                if len(self.steering_history) > 10:
                    self.steering_history.pop(0)

                # 실시간 표시
                progress = frame_idx / len(df) * 100
                wheel_display = self.draw_steering_wheel(predicted_steering, actual_steering)

                # 성능 상태
                if avg_error < 0.05:
                    status = "🟢 EXCELLENT"
                elif avg_error < 0.1:
                    status = "🟡 GOOD"
                else:
                    status = "🔴 LEARNING"

                # 화면 출력
                os.system('clear' if os.name == 'posix' else 'cls')  # 화면 클리어

                print("🚗 PilotNet 실시간 자율주행 시뮬레이터")
                print("=" * 80)
                print(f"📊 Progress: {progress:5.1f}% | Frame: {frame_idx:3d}/{len(df)}")
                print(f"🎯 Current Status: {status}")
                print()

                print("🎮 조향각 비교:")
                print(f"   예측값: {predicted_steering:7.4f} 🔴")
                print(f"   실제값: {actual_steering:7.4f} 🔵")
                print(f"   오  차: {error:7.4f}")
                print()

                print("🎯 조향 휠 시각화:")
                print("좌←                    |중앙|                    →우")
                print(f" {wheel_display}")
                print("└─────────────────────┼─────────────────────┘")
                print()

                print("📈 성능 지표:")
                print(f"   평균 오차: {avg_error:.4f}")
                print(f"   처리된 프레임: {frame_count}")

                if len(self.steering_history) >= 5:
                    recent_errors = [h['error'] for h in self.steering_history[-5:]]
                    recent_avg = np.mean(recent_errors)
                    print(f"   최근 5프레임 평균 오차: {recent_avg:.4f}")

                print()
                print("📝 최근 히스토리:")
                for i, h in enumerate(self.steering_history[-5:]):
                    print(f"   {i+1}: Pred={h['predicted']:6.3f}, Actual={h['actual']:6.3f}, Error={h['error']:.3f}")

                print("\n" + "=" * 80)
                print("⏹️  Ctrl+C를 눌러서 중단하세요")

                time.sleep(0.5)  # 0.5초 대기

        except KeyboardInterrupt:
            print("\n\n⏹️  시뮬레이션이 사용자에 의해 중단되었습니다.")
        except Exception as e:
            print(f"\n❌ 오류 발생: {e}")

        # 최종 결과
        print("\n🏁 시뮬레이션 완료!")
        print(f"📊 최종 성능:")
        print(f"   총 처리 프레임: {frame_count}")
        print(f"   평균 오차: {avg_error:.4f}")

        if avg_error < 0.05:
            print("🎉 훌륭한 성능입니다!")
        elif avg_error < 0.1:
            print("👍 좋은 성능입니다!")
        else:
            print("📚 더 많은 데이터로 학습하면 성능이 개선될 것입니다!")

# 텍스트 시뮬레이터 실행
simulator = TextPilotNetSimulator()
simulator.run_simulation()

In [ ]:
# 고급 PilotNet 시뮬레이터 스타일 영상 생성
import cv2
import numpy as np
import torch
import time

def create_advanced_pilotnet_video():
    # 모델 로드
    model = PilotNet().to(device)
    model.load_state_dict(torch.load('pilotnet_model.pth'))
    model.eval()

    # 데이터 로드
    df = pd.read_csv('data/driving_log.csv')

    # 비디오 설정 (고해상도)
    width, height = 1920, 1080  # Full HD
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter('pilotnet_simulator_style.mp4', fourcc, 15.0, (width, height))

    print("🎬 고급 PilotNet 시뮬레이터 영상 생성 중...")

    total_frames = len(df) // 2
    start_time = time.time()
    steering_history = []

    for i, frame_idx in enumerate(range(0, len(df), 2)):
        # 현재 이미지 로드
        img_path = f"data/{df.iloc[frame_idx]['center']}"
        image = cv2.imread(img_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 조향각 예측
        with torch.no_grad():
            processed = cv2.resize(image_rgb, (200, 66))
            processed = processed.astype(np.float32) / 255.0
            processed = np.transpose(processed, (2, 0, 1))
            processed = torch.FloatTensor(processed).unsqueeze(0).to(device)
            predicted_steering = model(processed).cpu().item()

        actual_steering = float(df.iloc[frame_idx]['steering'])

        # 조향각 히스토리 업데이트
        steering_history.append({
            'predicted': predicted_steering,
            'actual': actual_steering
        })

        if len(steering_history) > 100:  # 더 긴 히스토리
            steering_history.pop(0)

        # 시뮬레이터 스타일 프레임 생성
        frame = create_simulator_frame(image_rgb, predicted_steering, actual_steering,
                                     steering_history, frame_idx, total_frames, width, height)

        # BGR로 변환
        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        out.write(frame_bgr)

        # 진행률 표시
        if i % 20 == 0:
            progress = (i + 1) / total_frames * 100
            elapsed = time.time() - start_time
            if i > 0:
                remaining = elapsed * total_frames / (i + 1) - elapsed
                print(f"🎥 {progress:5.1f}% | {i+1:3d}/{total_frames} | 남은시간: {remaining:4.0f}초")

    out.release()
    total_time = time.time() - start_time
    print(f"✅ 완료! pilotnet_simulator_style.mp4 ({total_time:.0f}초 소요)")

def create_simulator_frame(image, pred_steering, actual_steering, history, frame_idx, total_frames, width, height):
    """시뮬레이터 스타일 프레임 생성"""

    # 배경 - 진한 파란색 (Udacity 스타일)
    canvas = np.full((height, width, 3), (20, 40, 80), dtype=np.uint8)

    # === 1. 상단 타이틀 ===
    title_text = f"PilotNet Autonomous Driving Simulator - Progress: {frame_idx/total_frames*100:.1f}% ({frame_idx}/{total_frames})"
    cv2.putText(canvas, title_text, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)

    # === 2. 메인 도로 영상 (오른쪽) ===
    main_img_width = 800
    main_img_height = 450
    main_x = width - main_img_width - 50
    main_y = 100

    # 이미지 크기 조정 및 배치
    main_img = cv2.resize(image, (main_img_width, main_img_height))
    canvas[main_y:main_y+main_img_height, main_x:main_x+main_img_width] = main_img

    # 도로 영상 테두리
    cv2.rectangle(canvas, (main_x-5, main_y-5),
                  (main_x+main_img_width+5, main_y+main_img_height+5), (255, 255, 255), 3)

    # 도로 영상 라벨
    cv2.putText(canvas, "Camera View", (main_x, main_y-15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # === 3. 조향각 게이지 (왼쪽 위) ===
    gauge_center_x = 300
    gauge_center_y = 300
    gauge_radius = 150

    # 외부 원
    cv2.circle(canvas, (gauge_center_x, gauge_center_y), gauge_radius, (200, 200, 200), 4)
    cv2.circle(canvas, (gauge_center_x, gauge_center_y), gauge_radius-20, (100, 100, 100), 2)

    # 눈금 표시
    for angle in range(-90, 91, 30):
        rad = np.radians(angle)
        inner_x = int(gauge_center_x + (gauge_radius-30) * np.sin(rad))
        inner_y = int(gauge_center_y - (gauge_radius-30) * np.cos(rad))
        outer_x = int(gauge_center_x + (gauge_radius-10) * np.sin(rad))
        outer_y = int(gauge_center_y - (gauge_radius-10) * np.cos(rad))
        cv2.line(canvas, (inner_x, inner_y), (outer_x, outer_y), (200, 200, 200), 2)

    # 예측 조향각 바늘 (빨간색)
    pred_angle_rad = np.radians(pred_steering * 90)  # -1~1 → -90~90도
    pred_end_x = int(gauge_center_x + (gauge_radius-40) * np.sin(pred_angle_rad))
    pred_end_y = int(gauge_center_y - (gauge_radius-40) * np.cos(pred_angle_rad))
    cv2.arrowedLine(canvas, (gauge_center_x, gauge_center_y), (pred_end_x, pred_end_y),
                   (0, 0, 255), 8, tipLength=0.3)

    # 실제 조향각 바늘 (초록색)
    actual_angle_rad = np.radians(actual_steering * 90)
    actual_end_x = int(gauge_center_x + (gauge_radius-60) * np.sin(actual_angle_rad))
    actual_end_y = int(gauge_center_y - (gauge_radius-60) * np.cos(actual_angle_rad))
    cv2.arrowedLine(canvas, (gauge_center_x, gauge_center_y), (actual_end_x, actual_end_y),
                   (0, 255, 0), 6, tipLength=0.3)

    # 중앙점
    cv2.circle(canvas, (gauge_center_x, gauge_center_y), 10, (255, 255, 255), -1)

    # 게이지 라벨
    cv2.putText(canvas, "Steering Angle", (gauge_center_x-80, gauge_center_y+gauge_radius+30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # === 4. 수치 정보 (왼쪽 아래) ===
    info_x = 50
    info_y = 600
    line_height = 40

    error = abs(pred_steering - actual_steering)

    # 정보 박스 배경
    cv2.rectangle(canvas, (info_x-20, info_y-20), (info_x+400, info_y+300), (40, 60, 100), -1)
    cv2.rectangle(canvas, (info_x-20, info_y-20), (info_x+400, info_y+300), (255, 255, 255), 2)

    info_lines = [
        f"Predicted Steering: {pred_steering:7.4f}",
        f"Actual Steering:    {actual_steering:7.4f}",
        f"Error:              {error:7.4f}",
        f"",
        f"Frame: {frame_idx}",
        f"Progress: {frame_idx/total_frames*100:.1f}%",
        f"",
        f"Status: {'EXCELLENT' if error < 0.05 else 'GOOD' if error < 0.1 else 'LEARNING'}"
    ]

    for i, line in enumerate(info_lines):
        y_pos = info_y + i * line_height
        color = (0, 0, 255) if 'Predicted' in line else (0, 255, 0) if 'Actual' in line else (255, 255, 255)
        cv2.putText(canvas, line, (info_x, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # === 5. 조향각 히스토리 그래프 (하단) ===
    if len(history) > 1:
        graph_x = 50
        graph_y = height - 200
        graph_width = width - 100
        graph_height = 120

        # 그래프 배경
        cv2.rectangle(canvas, (graph_x, graph_y), (graph_x+graph_width, graph_y+graph_height),
                     (30, 50, 70), -1)
        cv2.rectangle(canvas, (graph_x, graph_y), (graph_x+graph_width, graph_y+graph_height),
                     (255, 255, 255), 2)

        # 중앙선
        center_y = graph_y + graph_height // 2
        cv2.line(canvas, (graph_x, center_y), (graph_x+graph_width, center_y), (128, 128, 128), 1)

        # 데이터 플롯
        if len(history) > 1:
            x_scale = graph_width / len(history)
            y_scale = graph_height // 4  # -1~1 범위

            for i in range(1, len(history)):
                # 예측값 (빨간색)
                x1 = int(graph_x + (i-1) * x_scale)
                x2 = int(graph_x + i * x_scale)
                y1 = int(center_y - history[i-1]['predicted'] * y_scale)
                y2 = int(center_y - history[i]['predicted'] * y_scale)
                cv2.line(canvas, (x1, y1), (x2, y2), (0, 0, 255), 3)

                # 실제값 (초록색)
                y1_actual = int(center_y - history[i-1]['actual'] * y_scale)
                y2_actual = int(center_y - history[i]['actual'] * y_scale)
                cv2.line(canvas, (x1, y1_actual), (x2, y2_actual), (0, 255, 0), 3)

        # 그래프 라벨
        cv2.putText(canvas, "Steering History", (graph_x, graph_y-10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(canvas, "Red: Predicted | Green: Actual", (graph_x, graph_y+graph_height+25),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # === 6. 범례 (오른쪽 하단) ===
    legend_x = width - 300
    legend_y = height - 120

    cv2.putText(canvas, "Legend:", (legend_x, legend_y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(canvas, "Red Arrow: AI Prediction", (legend_x, legend_y+25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    cv2.putText(canvas, "Green Arrow: Human Driver", (legend_x, legend_y+45), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    return canvas

# 고급 영상 생성 실행
create_advanced_pilotnet_video()